This is a sliding window. It grabs 500 characters, saves that as a chunk, then moves forward by only 400 characters (500 − 100 overlap) — so the next chunk re-includes the last 100 characters of the previous one. Without overlap, a fact split exactly at a chunk boundary (like "...employees accrue 18 | days of PTO...") could lose its meaning in retrieval, since neither half alone captures the full fact.

In [25]:
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 100) -> list[str]:
    """
    Split text into overlapping chunks (measured in characters).
    chunk_size: how big each chunk is
    overlap: how much repeats between consecutive chunks, so meaning
             isn't lost right at a chunk boundary.
    """
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

In [26]:
sample = "A" * 1200  # fake 1200-character string
result = chunk_text(sample)
print(len(result), "chunks")
for c in result:
    print(len(c))

3 chunks
500
500
400


In [27]:
DATA_DIR = "../data"

with open(f"{DATA_DIR}/company_handbook.txt", "r") as f:
    text = f.read()

chunks = chunk_text(text)
print(f"{len(chunks)} chunks created")
print("---")
print(chunks[0])

6 chunks created
---
EMPLOYEE HANDBOOK - NovaTech Solutions

SECTION 1: REMOTE WORK POLICY
NovaTech employees may work remotely up to 3 days per week. Remote work requests must be approved by your direct manager at least 48 hours in advance. Employees working remotely are expected to be available on Slack during core hours of 10 AM to 4 PM local time. Equipment stipends of $500 per year are provided for home office setup, covering items like monitors, chairs, and keyboards.

SECTION 2: LEAVE POLICY
Full-time employe


In [28]:
import os

all_chunks = []
all_ids = []
all_metadata = []

for filename in os.listdir(DATA_DIR):
    if not filename.endswith(".txt"):
        continue

    with open(os.path.join(DATA_DIR, filename), "r") as f:
        text = f.read()

    file_chunks = chunk_text(text)
    print(f"{filename}: {len(file_chunks)} chunks")

    for i, chunk in enumerate(file_chunks):
        all_chunks.append(chunk)
        all_ids.append(f"{filename}-{i}")
        all_metadata.append({"source": filename, "chunk_index": i})

print(f"\nTotal chunks across all files: {len(all_chunks)}")

company_handbook.txt: 6 chunks

Total chunks across all files: 6


In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

class TfidfEmbedder:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(max_features=384)
        self.is_fitted = False

    def fit(self, corpus: list[str]):
        self.vectorizer.fit(corpus)
        self.is_fitted = True

    def _embed(self, input: list[str]) -> list[list[float]]:
        if not self.is_fitted:
            raise RuntimeError("Embedder must be fit before use.")
        vectors = self.vectorizer.transform(input).toarray()
        if vectors.shape[1] < 384:
            pad = np.zeros((vectors.shape[0], 384 - vectors.shape[1]))
            vectors = np.hstack([vectors, pad])
        return vectors.tolist()

    def __call__(self, input: list[str]) -> list[list[float]]:
        return self._embed(input)

    def embed_documents(self, input: list[str]) -> list[list[float]]:
        return self._embed(input)

    def embed_query(self, input) -> list[list[float]]:
        # Normalize to a list of strings, whatever shape chromadb passes in
        if isinstance(input, str):
            texts = [input]
        else:
            texts = input
        return self._embed(texts)

In [30]:
embedder = TfidfEmbedder()
embedder.fit(all_chunks)

# Try embedding just the first chunk to see what a vector looks like
sample_vector = embedder([all_chunks[0]])

print("Vector length:", len(sample_vector[0]))
print("First 10 numbers:", sample_vector[0][:10])
print("How many non-zero values:", sum(1 for v in sample_vector[0] if v != 0))

Vector length: 384
First 10 numbers: [0.06979799395301463, 0.0, 0.0, 0.0, 0.0, 0.11765177887281465, 0.0, 0.11765177887281465, 0.0, 0.0]
How many non-zero values: 63


In [31]:
import chromadb

DB_DIR = "../chroma_db"

client = chromadb.PersistentClient(path=DB_DIR)

# Fresh start each time we re-run ingestion, to avoid duplicate chunks
try:
    client.delete_collection("handbook")
except Exception:
    pass

collection = client.create_collection(name="handbook", embedding_function=embedder)

collection.add(
    documents=all_chunks,
    ids=all_ids,
    metadatas=all_metadata,
)

print("Stored", collection.count(), "chunks in the vector DB")

Stored 6 chunks in the vector DB


In [32]:
results = collection.query(
    query_texts=["how many vacation days do I get?"],
    n_results=2,
)

for doc, meta, dist in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    print(f"[{meta['source']} | distance={dist:.3f}]")
    print(doc[:150])
    print("---")

[company_handbook.txt | distance=1.466]
tup, covering items like monitors, chairs, and keyboards.

SECTION 2: LEAVE POLICY
Full-time employees accrue 18 days of paid time off (PTO) per year,
---
[company_handbook.txt | distance=1.570]
PTO up to 5 days may be carried over to the following year; anything beyond that is forfeited.

SECTION 3: EXPENSE REIMBURSEMENT
Employees may submit 
---
